# RAG - Retrieval, Scalability from Scratch

- Exact search vs Approximate search
- Graph
- FAISS

Pipeline:

1. Documents
2. Chunk documents
3. Embedding
4. FAISS indexing
5. Query
6. Embedding (query)
7. ANN Search in FAISS index
8. Top-k chunks
9. LLM context
10. Response

## Facebook AI Similarity Search (FAISS)

A library including **indexing** methods:

- Flat Index
- IVF
- PQ
- OPQ
- HNSW
- IVF+PQ
- GPU search
- Binary indexes

### Flat Index

Index every vector, no approximation!

> Only use on small number of vectors (1,000)

But when we have billions, forget it.

In [2]:
from src import cosine_similarity

vectors = []
query = []

for vector in vectors:
    cosine_similarity(query, vector)

To overcome huge number of vectors, we can:

- Partition the space -> IVF
- Compress vectors -> PQ
- Use GPU
- Combine methods

### Inverted File Index (IVF)

- Run **clustering** > **store** vectors.
- k-means > centroids
- Search nearest centroid > inspect nearby vectors.

> But the nearest neighbour might not in the searched cluster.

Then FAISS allows search multiple clusters.

### Product Quantisation (PQ)

- Store tiny codes instead of values (e.g., store 64 bytes, not 1536 numbers).
- Lose information but fast.

### GPU

- Simply use GPU for computations.

### Combine Methods

- Google-scale systems:

> Query > Embedding > IVF > HNSW inside cluster > PQ compression > Top 100 candidates > Exact cosine similarity > Top 10 documents.

- Approximation first, exact later.

## Build from Scratch

1. Build a simple IVF index from scratch

- Implement k-means.
- Assign vectors to clusters.
- Search only selected clusters.
- Measure the speed/accuracy trade-off against linear search.

2. Build a simple Product Quantisation prototype

- Split vectors into subvectors.
- Learn small codebooks.
- Compress vectors.
- Compare memory usage and retrieval quality.

3. Only then use FAISS

- Understand `IndexFlatL2`, `IndexFlatIP`, `IndexIVFFlat`, `IndexHNSWFlat`, `IndexIVFPQ`, and why each exists.
- You'll recognise them as implementations of concepts you've already built rather than mysterious APIs.

### IVF Index From Scratch

- Clustering
- Centroids
- k-means

#### K-means

1. Random centroids
2. Assign points to clusters
3. Update centroids with mean of dimensions

Formula:

```math
\sum_{i=1}^K \sum_{x \in C_i}||x - \mu_i||^2
```

- $C_i$ := set of points in cluster i
- $\mu_i$ := centroid (mean) of that cluster

First, build KMeans class for the formula.

- `centroids`: stores coordinates of centroids, `shape = (k, embedding_dimension)`.
- `labels`: `np.array` type, tells which cluster every vector belongs to, parallel to the vector list.

Think of algorithm, we have:

Random centroids > Assign every points > Move centroids > Repeat.

The class also have these methods:

- init centroids
- assign clusters
- update centroids
- fit

In [ ]:
class KMeans:

    def __init__(self, k: int):
        self.k = k
        self.centroids = None
        self.labels = None

    def initialise_centroids():
        ...
    
    def assign_clusters(self, vectors: np.ndarray):
        '''
        Assign input vector to a centroid.
        '''
        if self.centroids is None:
            return "No centroids created."
        
        self.labels = np.empty(len(vectors), dtype=int)
        
        for i in range(len(vectors)):
            max_score = -np.inf
            best_centroid = None

            for j, centroid in enumerate(self.centroids):
                score = cosine_similarity(vectors[i], centroid)
                if score > max_score:
                    max_score = score
                    best_centroid = j
            
            self.labels[i] = best_centroid

    def update_centroids(self):
        '''
        Average points in a cluster to get the actual centroid.
        '''
        ...

    def fit(self, vectors):
        '''
        Once centroids are moved, they change which vectors belong to them.
        We have to do the process again and again till Convergence.
        '''
        ...

#### Inverted File

Instead of a big list of vectors, we keep an inverted mapping from each centroid to the vectors assigned to it.

- Centroid 0 -> list of vector IDs
- Centroid 1 -> list of vector IDs
- ...


**Offline indexing**

```python
class IVF:

    def __init__(self):
        self.centroids: np.array
        self.inverted_list
    
    def fit():
        ...
    
    def search():
        ...
```

An IVF object needs only store list of vector IDs, each of which stores `chunk` and `embedding`.

But for learning purpose, each inverted list stores:

```python
class Posting:

    chunk_id
    text
    embedding
```

**Algorithm**

Fit vectors to clusters with K-Means.

```
IVF-FIT(vectors)
    centroids, labels <- K-MEANS.FIT(vectors)
    
    for i in labels
        ivf.list[labels[0]].append(vectors[i])
```

Search for vectors

```
IVF-SEARCH(query_embedding)
    Compare query_embedding with centroids
    Get the nearest centroid
    Get all Posting in the centroid
    Compare query_embedding with each Posting
    Return top-k
```

**Online Search**

1. Query
2. Query embedding
3. Nearest centroid
4. Retrieve inverted list
5. Compute similarity
6. Top k

> But what if the nearest document is in another cluster?

That's why we need FAISS's `nprobe`, instead of searching 1 centroid, we search k nearest centroids.

#### Full Flow

Query > Nearest Centroid > Retrieve assigned vectors > Cosine Similarity > Top K.

## Product Quantisation

Vectors are stored as `float32`. Imagine a vector has 768 dimensions, then:

```math
768 * 4 \text{ bytes} = 3072 \text{ bytes/vector} = 3 { KB}
```

Then 100 milion vectors is 300 GB.

-> Store a tiny compressed representation = **Product Quantisation**.

- 1 vector (dimension `d`) -> m Subvectors (or *Subspace*) (dimension = `d/m`)
- each Subspace: K-Means -> Codebook (k centroids) -> Codeword (centroid)

> **Compression** (FAISS): Separate 1 vector > Subspaces > K-Mean get the nearest centroids (codebook) > Store only the codebooks instead of the subvectors.

For search, only need to get one nearest centroid.

In [ ]:
class ProductQuantiser:

    def __init__(self, m, k):
        self.m = m
        self.k = k
        self.subvector_dim = None
        self.codebooks = []
    
    def fit(self, vectors):
        '''
        - Split vectors
        - K-Means
        - Store codebooks
        '''
        ...
    
    def encode(self, vectors):
        '''
        Turn vectors to codebooks
        '''
    
    def decode(self, code):
        '''
        - Look up centroid
        - Concatenate
        - Approximate vector
        '''

## Real FAISS

**Problem**

Comparing 1 million vectors `q` with `v1`, `v2`,..., `v1000000`

1. Level 1 - Exact Search

```python
for vector in vectors:
    score = similarity(query, vector)
```

This is also called **Flat Search**, all vectors are stored in one *big array*. FAISS calls these `IndexFlat`.

Then we have `IndexFlatL2` computing *Euclidean Distance*.

```python
for vector in vectors:
    dist = euclidean_distance(q - vector)**2 
```

Then we have `IndexFlatIP`, which uses *inner product* instead, the same as cosine similarity.

2. Level 2 - IVF

FAISS names this `IndexIVFFlat` (= IVF + Flat vectors).

- `Flat`: stores real vectors, no compression.

3. Level 3 - IVFPQ (= IVF + PQ)

Instead of raw vectors, store PQ codes (Product Quantisation)

4. Level 4 - HNSW

- No KMeans, no centroids, no clusters.
- Use graph: Random node > Better neighbour > repeat

FAISS has `IndexHNSWFlat` (= Graph search + real vectors) no compression.